# Imports

In [1]:
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from transformers import BertModel, BertConfig
from torch.utils.data import DataLoader

# Data handeling
from src.data.dataset import MLMDataset, ClassificationDataset
from src.data.data_tools import filter_taxonomy, fasta2pandas

#Vocab shit
from src.utils.vocab import Vocabulary, KmerVocabConstructor

# Preprocessing
from src.preprocessing.augmentation import SequenceModifier, IdentityStrategy, BaseStrategy
from src.preprocessing.tokenization import KmerStrategy
from src.preprocessing.padding import PEndStrategy
from src.preprocessing.truncation import TEndStrategy
from src.preprocessing.preprocessor import Preprocessor

# Model things
from src.model.backbone import Bertax, ModularBertax
from src.model.encoders import LabelEncoder
from src.model.heads import MLMHead, SingleClassHead

# Training things
from src.train.trainers import MLMtrainer, ClassificationTrainer

In [2]:


CONFIG = {
    "FILE_PATH": "src/data/raw.fasta",
    "SAVE_PATH": "pretrained_single_model_k2.pt",
    "n_test": 10,
    "modification_probability": 0.05,
    "alphabet": ["A", "C", "G", "T"],
    "k": [2, 3, 4],
    "optimal_length": 600,

    # Training parameters
    "num_epochs": 100,
    "masking_percentage": 0.05,
    "batch_size": 386,
    "small_set": False,

    # Model configuration
    "num_layers": 10,
    "num_attention_heads": 4,
    "hidden_size": 256,
    "intermediate_size": 1024,  # 4 * hidden_size
    "dropout_rate": 0.05,
    "num_classes": 19,
    "mlm_dropout_rate": 0.1,

    #Classification 
    "target_label": "genus"
}


In [3]:

def set_up_preprocessors(CONFIG):
    train_preprocessors = []
    val_preprocessors = []

    for k in CONFIG["k"]:
        # Set up vocabulary
        constructor = KmerVocabConstructor(k=k, alphabet=CONFIG["alphabet"])
        vocab = Vocabulary()
        vocab.build_from_constructor(constructor, data=[])
        vocab_path = f"vocab_{k}.json"
        vocab.save(vocab_path)

        # Set up preprocessors
        sequence_modifier = SequenceModifier(alphabet=CONFIG["alphabet"])
        augmentation_strategy_train = BaseStrategy(
            modifier=sequence_modifier,
            alphabet=CONFIG["alphabet"],
            modification_probability=CONFIG["modification_probability"]
        )

        augmentation_strategy_val = IdentityStrategy(
            modifier=sequence_modifier,
            alphabet=CONFIG["alphabet"],
            modification_probability=0
        )

        tokenization_strategy = KmerStrategy(
            k=k,
            padding_alphabet=CONFIG["alphabet"]
        )

        padding_strategy = PEndStrategy(
            optimal_length=CONFIG["optimal_length"]//k
        )

        truncation_strategy = TEndStrategy(
            optimal_length=CONFIG["optimal_length"]//k
        )

        preprocessor_train = Preprocessor(
            augmentation_strategy=augmentation_strategy_train,
            tokenization_strategy=tokenization_strategy,
            padding_strategy=padding_strategy,
            truncation_strategy=truncation_strategy,
            vocab=vocab,
        )

        preprocessor_val = Preprocessor(
            augmentation_strategy=augmentation_strategy_val,
            tokenization_strategy=tokenization_strategy,
            padding_strategy=padding_strategy,
            truncation_strategy=truncation_strategy,
            vocab=vocab,
        )

        train_preprocessors.append(preprocessor_train)
        val_preprocessors.append(preprocessor_val)
    return train_preprocessors, val_preprocessors

train_preprocessors, val_preprocessors = set_up_preprocessors(CONFIG)

In [4]:
##################################################################
## Data preparation ##############################################
##################################################################

all_data = fasta2pandas(CONFIG["FILE_PATH"])

#tmp: to use a smaller set for test if code compiles - nothing to be used in production
if CONFIG["small_set"]:
    all_data = all_data[:CONFIG["n_test"]]

# filter data for finetuning
filtered_data = filter_taxonomy(
    df = all_data,
    startAt ='phylum',
    endAt = 'species',
    phylumCertainty=True
    )

# 
num_classes = len(list(set(filtered_data[CONFIG["target_label"]])))

# encodes label for classification
label_encoder = LabelEncoder(filtered_data[CONFIG["target_label"]])

# Pretraining datasplit based on unsplit data: 
pretrain_sequences, preval_sequences = train_test_split(
    all_data["sequence"],
    test_size = 0.1,
    random_state = 42
)

# Finetune datasplit based on filtered data:
finetrain_data, fineval_data = train_test_split(
    filtered_data,
    test_size = 0.1,
    random_state = 69
    )

print(f"Number of pre-training sequences: {len(pretrain_sequences)}")
print(f"Number of validation sequences: {len(preval_sequences)}")

Applying filters...
Filtering complete.

Handling DNA ambiguity codes...
Processing row 0...
Processing row 10000...
Processing row 20000...
Processing row 30000...
Processing row 40000...
Processing row 50000...
Processing row 60000...
Processing row 70000...
Processing row 80000...
Processing row 90000...
Processing complete.
Number of pre-training sequences: 83962
Number of validation sequences: 9330


In [5]:
test_sequence = pretrain_sequences[0]
print(f"Test sequence: {test_sequence}")
print()

for processor in train_preprocessors:
    output = processor.process(test_sequence)
    print(f"Output is: {output}")
    print(f"Length of output is {len(output)}")
    print()


Test sequence: AAGGATCATTATCGAGTGGGGGTCCTCTGGGCCCCGTCTCCAACCCTTGTCTATTCTACCATGTTGCTTTGGCGGGCCCGTCTGCAACCGGACCGCTGGGGACTCGCGCCCCTGGCCCGCGCCCGTCAATAGCCCCCCCAACTTTTTCTAACAGTGACGTCTAAGCAAACGAGAATAACCAAAACTTTCAACAACGGATCTCTTGGTTCTGGCATCGATGAAGAACGCAGCGAAATGCGATAAGTAATGCGAATTGCAGAATTCCGTGAGTCATCGAATCTTTGAACGCACATTGCGCCCTCTGGTACTCCGGAGGGCATGCCTGTTCGAGCGTCATTGTCAACCGTCAAGCTCGGCTTGCTGTTGGGTCCCCGTCGTTGCTCCAGCGGCGGACCCGAAAGATAATGGCAGAGTCTGTGAGACCCTGGATGCAGCGAGCTTCTAGCACGCGTCTGGACGGTCTTTAGGCTTGGTCTCAACCAGTTGTCAACTTCTG

Output is: [5, 15, 8, 9, 20, 8, 11, 7, 19, 15, 15, 18, 12, 12, 15, 14, 10, 11, 12, 10, 5, 10, 12, 19, 18, 17, 20, 6, 17, 10, 8, 16, 19, 12, 20, 15, 11, 15, 10, 11, 18, 19, 9, 6, 11, 13, 10, 14, 19, 15, 13, 12, 11, 11, 10, 10, 19, 14, 10, 11, 10, 11, 18, 5, 17, 14, 10, 10, 10, 5, 12, 20, 20, 12, 5, 9, 16, 13, 11, 18, 17, 7, 9, 5, 11, 7, 5, 17, 6, 9, 5, 12, 20, 9, 6, 5, 11, 13, 18, 18, 20, 15, 20, 12, 15, 9, 18, 13, 13, 7, 5, 11, 11, 11, 5, 8, 14, 13, 17, 7, 17, 8, 14, 16, 5, 20, 14, 

In [6]:
from torch.utils.data import Dataset

class MultipleKDataset(Dataset):
    def __init__(self, CONFIG, df, preprocessors, label_encoder):
        self.CONFIG = CONFIG
        self.df = df
        self.preprocessors = preprocessors  
        self.target_column = CONFIG["target_label"]
        self.label_encoder = label_encoder  
    
    def __len__(self):
        return len(self.df)

    def _create_attention_mask(self, input_seq, preprocessor):
        """
        Returns an attention mask for the input sequence:
         - 1 where token != PAD
         - 0 where token == PAD
        """
        pad_id = preprocessor.vocab.get_id("PAD")
        return [1 if token != pad_id else 0 for token in input_seq]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sequence = row["sequence"]

        # Create dictionaries for the outputs
        input_ids_dict = {}
        attention_mask_dict = {}

        # Loop through each preprocessor and its corresponding k value.
        # Assuming the order in CONFIG["k"] matches the order in self.preprocessors.
        for i, preprocessor in enumerate(self.preprocessors):
            k_value = self.CONFIG["k"][i]
            preprocessed_sequence = preprocessor.process(sequence)
            attention_mask = self._create_attention_mask(preprocessed_sequence, preprocessor)
            
            # Store as tensors for each k
            input_ids_dict[str(k_value)] = torch.tensor(preprocessed_sequence, dtype=torch.long)
            attention_mask_dict[str(k_value)] = torch.tensor(attention_mask, dtype=torch.long)
        
        label = row[self.target_column]
        encoded_label = self.label_encoder.encode(label)

        return {
            "original_seq": sequence,
            "input_ids": input_ids_dict,
            "attention_masks": attention_mask_dict,
            "label": encoded_label,
            "encoded_label": encoded_label
        }


In [7]:

# Finetuning datasets
finetrain_dataset = MultipleKDataset(
    CONFIG = CONFIG,
    df = finetrain_data,
    preprocessors = train_preprocessors,
    label_encoder = label_encoder,
    )

fineval_dataset = MultipleKDataset(
    CONFIG = CONFIG,
    df = fineval_data,
    preprocessors = val_preprocessors,
    label_encoder = label_encoder,
    )

In [8]:
testdata = finetrain_dataset.__getitem__(1)
print(testdata)

for key in testdata["attention_masks"].values():
    print(len(key))

{'original_seq': 'ACCATATGAATGCGGACCGGGAATCCCAAAAATATCTGCCTTGCCTCTTTGGTAGGCGTATTTTTGCCCCGTCTGTTTGAATATTCACCCATGTCTTTTGCGTACTATTTGTTTCCTTGGTGGGTTCGCCCGCCAATAGGACACCATAAAACCTTTTGTAATTGCAGTCAGCGTCAGAAAAACTTAATAGTTACAACTTTCAACAACGGATCTCTTGGTTCTGGCATCGATGAAGAACGCAGCGAAATGCGATAAGTAGTGTGAATTGCAGAATTCAGTGAATCATCGAATCTTTGAACGCACATTGCGCCCCTTGGTATTCCATGGGGCATGCCTGTTCGAGCGTCATTTGTACCCTCAAGCCTTGCTTGGTGTTGGGTGTTTGTCTTCATCACTGGAGACTCGCCTTAAAACAATTGGCAGCCGGCATATTGGTCTTGGAGCGCAGCACAATTTGCGCTTCTTTCCATGAATGCTAGCGTCCATAAAGCCTATTTTAAC', 'input_ids': {'2': tensor([ 6,  9, 17, 19,  5, 19, 11, 13, 10, 15, 13,  8, 10,  9,  7,  5, 17, 18,
        19, 10, 20, 14, 12,  6, 20, 19, 16,  7, 11, 17, 20, 20, 19, 10, 10, 16,
        12, 16, 20, 13,  8,  8, 18,  6, 10,  8, 16, 12, 20, 19, 11, 17, 12,  8,
        20, 16, 20, 10, 20, 15, 19, 15, 20, 11, 10, 11, 10,  5, 17, 15,  6,  6,
         8,  5,  6,  6, 20, 20, 16,  5, 20, 14,  7, 18,  7, 11, 18,  7,  5,  5,
        12, 17,  8,  7, 20,  6,  5, 12, 20,  9,  6,  5, 11,

In [9]:
import torch.nn as nn
from transformers import BertModel, BertConfig
import torch


class MultipleKModularBertax(nn.Module):
    def __init__(self, encoders, fusion_encoder, classification_head):
        super(MultipleKModularBertax, self).__init__()

        self.encoders = nn.ModuleList(encoders)
        self.fusion_encoder = fusion_encoder
        self.classification_head = classification_head

    def forward(self, input_ids, attention_masks):
        pooled_outputs = []
        sorted_keys = sorted(input_ids.keys(), key=int)
        for idx, key in enumerate(sorted_keys):
            encoder_outputs = self.encoders[idx](
                input_ids=input_ids[key],
                attention_mask=attention_masks[key]
            )
            pooled_outputs.append(encoder_outputs.pooler_output)  # shape: [batch_size, hidden_size]

        # Stack CLS tokens to form a sequence: [batch_size, num_encoders, hidden_size]
        sequence_outputs = torch.stack(pooled_outputs, dim=1)
        
        # Pass this sequence to the fusion encoder
        fusion_outputs = self.fusion_encoder(
            inputs_embeds=sequence_outputs,
            attention_mask=torch.ones(sequence_outputs.shape[:2], device=sequence_outputs.device)
        )
        
        # Use the fusion encoder's pooled output for classification
        pooled_fusion = fusion_outputs.pooler_output
        classification_output = self.classification_head(pooled_fusion)
        return classification_output




In [10]:
# Setting up the model

# Setting up encoders
encoders = []
for k in CONFIG["k"]:
    vocab = Vocabulary()
    vocab.load(f"vocab_{k}.json")

    encoder_config = BertConfig(
        vocab_size = len(vocab),
        hidden_size = CONFIG["hidden_size"],
        num_hidden_layers = CONFIG["num_layers"],
        num_attention_heads = CONFIG["num_attention_heads"],
        intermediate_size = CONFIG["intermediate_size"],
        max_position_embeddings = CONFIG["optimal_length"] // k + 2,
        hidden_dropout_prob = CONFIG["dropout_rate"],
        attention_probs_dropout_prob = CONFIG["dropout_rate"]
    )

    encoder = BertModel(encoder_config)
    encoders.append(encoder)

fusion_encoder_config = BertConfig(
        vocab_size = len(vocab),
        hidden_size = CONFIG["hidden_size"],
        num_hidden_layers = CONFIG["num_layers"],
        num_attention_heads = CONFIG["num_attention_heads"],
        intermediate_size = CONFIG["intermediate_size"],
        max_position_embeddings = len(CONFIG["k"]) + len(CONFIG["k"]) + 1,
        hidden_dropout_prob = CONFIG["dropout_rate"],
        attention_probs_dropout_prob = CONFIG["dropout_rate"]
    )

fusion_encoder = BertModel(fusion_encoder_config)
classification_head = SingleClassHead(
    in_features = CONFIG["hidden_size"],
    hidden_layer_size = (CONFIG["hidden_size"] + CONFIG["num_classes"])//2,
    out_features = num_classes,
    dropout_rate = CONFIG["mlm_dropout_rate"]
)

model = MultipleKModularBertax(
    encoders = encoders, 
    fusion_encoder = fusion_encoder,
    classification_head = classification_head)

In [11]:
sample = finetrain_dataset.__getitem__(1)

# Unpack the dictionaries
input_ids = sample["input_ids"]
attention_masks = sample["attention_masks"]

# Add a batch dimension to each tensor in the dictionaries
for key in input_ids:
    input_ids[key] = input_ids[key].unsqueeze(0)       # Shape becomes [1, sequence_length]
for key in attention_masks:
    attention_masks[key] = attention_masks[key].unsqueeze(0)

# Now pass these batched inputs to your model
output = model(input_ids, attention_masks)

print(output)
print(len(output[0]))

tensor([[ 0.0126,  0.3100,  0.0649,  ...,  0.0925,  0.0473, -0.0151]],
       grad_fn=<AddmmBackward0>)
5855


In [12]:
def multiK_collate_fn(samples):
    batch = {}
    # For keys like "original_seq", "label", etc., stack or collect as usual.
    batch["original_seq"] = [s["original_seq"] for s in samples]
    batch["label"] = torch.tensor([s["label"] for s in samples])
    batch["encoded_label"] = torch.tensor([s["encoded_label"] for s in samples])
    
    # For dictionaries in "input_ids" and "attention_masks", do:
    input_ids_batch = {}
    attention_masks_batch = {}
    
    # Assume all samples have the same keys.
    keys = samples[0]["input_ids"].keys()
    for key in keys:
        input_ids_batch[key] = torch.stack([s["input_ids"][key] for s in samples], dim=0)
        attention_masks_batch[key] = torch.stack([s["attention_masks"][key] for s in samples], dim=0)
    
    batch["input_ids"] = input_ids_batch
    batch["attention_masks"] = attention_masks_batch
    
    return batch

finetrain_loader = DataLoader(
    dataset=finetrain_dataset,
    batch_size=CONFIG["batch_size"],
    collate_fn=multiK_collate_fn,
    shuffle=True
    )

fineval_loader = DataLoader(
    dataset=fineval_dataset,
    batch_size=CONFIG["batch_size"],
    collate_fn=multiK_collate_fn,
    shuffle=True
    )

In [13]:
class MultiKSingleClassificationTrainer:
    def __init__(self,
                 model: nn.Module,
                 train_loader: DataLoader,
                 val_loader: DataLoader,
                 weight_save_path: str = "best_classification_weights.pt"):
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        print(f"Training on device {self.device} and it is awesome!!!")
        
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(self.model.parameters(), lr=5e-6)
        self.best_val_loss = float('inf')
        self.weight_save_path = weight_save_path

    def _run_epoch(self, epoch_nr):
        self.model.train()
        total_loss, correct, total = 0, 0, 0
        progress_bar = tqdm(self.train_loader, desc=f"Training Epoch {epoch_nr + 1}", leave=False)

        for batch in progress_bar:
            # Get the dictionaries from the batch.
            input_ids = batch["input_ids"]
            attention_masks = batch["attention_masks"]
            labels = batch["encoded_label"].to(self.device)

            # Move each tensor in the dictionaries to the device.
            for key in input_ids:
                input_ids[key] = input_ids[key].to(self.device)
            for key in attention_masks:
                attention_masks[key] = attention_masks[key].to(self.device)

            self.optimizer.zero_grad()
            logits = self.model(input_ids, attention_masks)
            loss = self.criterion(logits, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += labels.size(0)
            progress_bar.set_postfix(loss=loss.item())

        return total_loss / len(self.train_loader), correct / total

    def _validate_epoch(self, epoch_nr):
        self.model.eval()
        total_loss, correct, total = 0, 0, 0
        progress_bar = tqdm(self.val_loader, desc=f"Validation Epoch {epoch_nr + 1}", leave=False)

        with torch.no_grad():
            for batch in progress_bar:
                input_ids = batch["input_ids"]
                attention_masks = batch["attention_masks"]
                labels = batch["encoded_label"].to(self.device)

                for key in input_ids:
                    input_ids[key] = input_ids[key].to(self.device)
                for key in attention_masks:
                    attention_masks[key] = attention_masks[key].to(self.device)

                logits = self.model(input_ids, attention_masks)
                loss = self.criterion(logits, labels)
                total_loss += loss.item()
                correct += (logits.argmax(dim=-1) == labels).sum().item()
                total += labels.size(0)
                progress_bar.set_postfix(loss=loss.item())

        return total_loss / len(self.val_loader), correct / total

    def train(self, num_epochs=10):
        for epoch_nr in range(num_epochs):
            train_loss, train_acc = self._run_epoch(epoch_nr)
            val_loss, val_acc = self._validate_epoch(epoch_nr)

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.weight_save_path)

            print(f"Epoch {epoch_nr + 1}/{num_epochs}")
            print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")
            print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")


In [14]:

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from tqdm import tqdm

classification_trainer = MultiKSingleClassificationTrainer(
    model = model,
    train_loader = finetrain_loader,
    val_loader = fineval_loader,
    weight_save_path = "best_classification_weights.pt"
)

Training on device cuda and it is awesome!!!


In [15]:
classification_trainer.train(CONFIG["num_epochs"])

Epoch 1/100
Train Loss: 8.6093, Train Accuracy: 0.0151
Val Loss: 8.5640, Val Accuracy: 0.0474


Epoch 2/100
Train Loss: 8.5163, Train Accuracy: 0.0419
Val Loss: 8.4694, Val Accuracy: 0.0506


KeyboardInterrupt: 

In [71]:
batch = next(iter(finetrain_loader))
print("Batch keys:", batch.keys())
for key in batch["input_ids"]:
    print(f"input_ids[{key}].shape: {batch['input_ids'][key].shape}")
for key in batch["attention_masks"]:
    print(f"attention_masks[{key}].shape: {batch['attention_masks'][key].shape}")
print("encoded_label.shape:", batch["encoded_label"].shape)


Batch keys: dict_keys(['original_seq', 'label', 'encoded_label', 'input_ids', 'attention_masks'])
input_ids[2].shape: torch.Size([768, 300])
input_ids[3].shape: torch.Size([768, 200])
input_ids[4].shape: torch.Size([768, 150])
attention_masks[2].shape: torch.Size([768, 300])
attention_masks[3].shape: torch.Size([768, 200])
attention_masks[4].shape: torch.Size([768, 150])
encoded_label.shape: torch.Size([768])
